# Comparing localization functionals — MV, Stengel-Spaldin, Pipek-Mezey

`core.optim.minimize_spread` doesn't have to minimize the ordinary Marzari-Vanderbilt (MV) spread $\Omega$ — any differentiable $\Omega(U)$ works, since every optimizer just autodiffs whatever forward function it's handed. This notebook compares three:

- **MV** (Marzari & Vanderbilt, PRB **56**, 12847 (1997)) — the default everywhere else in this project: minimize the total spread $\sum_n \langle r^2\rangle_n - \langle r\rangle_n^2$.
- **SS** (Stengel & Spaldin, PRB **73**, 075121 (2006)) — an alternative $\Omega_D$ (a k-averaged variance of $\tilde M_{nn}$ instead of MV's branch-cut-sensitive $-\mathrm{Im}\ln \tilde M_{nn}$ residual); $\Omega_I$/$\Omega_{OD}$ are identical to MV's own (`use_ss_functional=True`, already used by tutorial36).
- **PM** (Pipek & Mezey, *J. Chem. Phys.* **90**, 4916 (1989)), NEW this notebook — a completely different localization *criterion*: instead of minimizing real-space spread, **maximize atomic character** — $\Omega_\mathrm{PM} = -\sum_n\sum_A Q_A[n]^2$, where $Q_A[n]$ is Wannier function $n$'s Mulliken charge on atom $A$ (built from `pw2wannier90`'s `atom_proj_ext` atomic-pseudo-orbital overlaps, `core.spread.compute_pm_spread` — new capability). PM orbitals aren't necessarily compact in space at all; they're compact in *which atoms* they sit on.

**Not implemented**: Edmiston-Ruedenberg (maximize each orbital's Coulomb self-repulsion $\iint |w_n(r)|^2|w_n(r')|^2/|r-r'|\,dr\,dr'$) — a real, classic alternative, but it needs a genuine real-space/reciprocal-Coulomb-kernel machinery this project's overlap-matrix-only ($M_{mn}$/$A_{mn}$) pipeline doesn't have; not attempted rather than shipped half-validated.

In [ ]:
import os, sys, pathlib
import numpy as np
import matplotlib.pyplot as plt

HERE = pathlib.Path.cwd()
REPO = HERE
while not (REPO / 'waw').exists() and REPO != REPO.parent:
    REPO = REPO.parent
sys.path.insert(0, str(REPO))

import waw
from waw.interfaces import quantum_espresso as qe   # the direct-input QE driver
from waw.interfaces.ase.driver import wannierize
from waw.units import BOHR_TO_ANG, HARTREE_TO_EV, EV_TO_HARTREE
from waw.vis import plot_bands, BandSeries

PSEUDO_DIR = REPO / 'workflows' / 'pseudos'   # shared across workflows/, not notebooks/-specific
NCORES = 16
waw.set_num_threads(NCORES)
print('waw', 'threads =', waw.get_num_threads(), '| repo', REPO)

### 1. Structure, DFT, and atomic projectors — same recipe as
`w90tutorial/35_silicon_atom_proj_ext`: bulk diamond Si, `atom_proj_ext=True` reads the pseudopotential's own $s$/$p$ pseudo-atomic-orbitals back out as the trial projections, giving `Aat` — the SAME tensor PM's Mulliken charges are built from — for free alongside the ordinary `Amn`. `proj_min`/`proj_max` (projectability) plus a loose energy window disentangle 40 candidate bands down to 8 target MLWFs (4 valence + 4 low conduction, matching the s+p atomic character of 2 Si atoms).

In [ ]:
from ase.build import bulk
import torch
from waw.core.disentangle import disentangle
from waw.core.init import svd_init
from waw.core.spread import rotate_overlaps, compute_pm_spread
from waw.core.optim import minimize_spread
from waw.interfaces.ase.driver import build_wannier_data
from waw.interfaces.ase.structure import recip_lattice
from waw.interfaces.quantum_espresso.upf import (
    read_pswfc, write_atom_proj_ext, atom_proj_column_atoms)

atoms = bulk('Si', 'diamond', a=5.43)
MP_GRID = (6, 6, 6)
WORK = HERE / 'runs' / 'functional_compare'
EXT_DIR = WORK / 'ext_proj'

radial = read_pswfc(PSEUDO_DIR / 'Si.upf')
write_atom_proj_ext(EXT_DIR, {'Si': radial})
atom_index = torch.tensor(atom_proj_column_atoms(atoms, {'Si': radial}))
print('l channels in Si.upf:', sorted(radial), '-> atom_index:', atom_index.tolist())

ov = qe.generate_overlaps(
    atoms, MP_GRID, WORK, 'si_pm',
    ecutwfc=40, scf_kpts=(8, 8, 8), nbnd=40, num_wann=8,
    atom_proj_ext=True, atom_proj_dir=EXT_DIR,
    pseudopotentials={'Si': 'Si.upf'}, pseudo_dir=PSEUDO_DIR, ncores=NCORES,
    rerun_scf=False,
)
print('overlaps ready, nk =', len(ov['kpts']))

### 2. Disentangle once, then hand the SAME initial gauge to every functional
Same reasoning as the optimizer-comparison notebook: an apples-to-apples comparison needs a shared `U_init`, not three independent Wannierisations that happen to start from nominally equal subspaces. `Aat_sub` is `Amn` (here, atomic-orbital overlaps, not SCDM/analytic trial functions) projected into the disentangled subspace the same way `Mmn_opt` is.

In [ ]:
wdata = build_wannier_data(
    recip_lattice(atoms), ov['kpts'], ov['mmn'], ov['amn'], ov['eig'],
    ov['nnkpts'], ov['g_vectors'],
)
dis = disentangle(
    wdata.Mmn, wdata.eig, wdata.wb, wdata.kb_idx, nw=8, Amn=wdata.Amn,
    proj_min=0.01, proj_max=0.95, frozen_window=(-1.0e6, 8.7 * EV_TO_HARTREE),
    n_iter=1000, conv_tol=1e-10,
)
print(f'Omega_I = {dis.omega_i * BOHR_TO_ANG**2:.4f} Ang^2  (converged: {dis.converged})')

Mmn_opt = rotate_overlaps(dis.V, wdata.Mmn, wdata.kb_idx)
Aat_sub = torch.einsum('kmi,kmj->kij', dis.V.conj(), wdata.Amn)   # atomic-orbital overlaps, same subspace
U_init = svd_init(Aat_sub)
print('U_init:', U_init.shape)

### 3. Optimize MV, SS, and PM from the same `U_init`
All three via `optimizer='cg'` (this project's own default); PM's `Omega`/centres in the returned `SpreadResult` are still the ordinary MV ones evaluated at the PM-optimized `U_final` (PM has no spread decomposition of its own), so the same `.Omega` field is a fair like-for-like comparison across all three.

In [ ]:
COMMON = dict(optimizer='cg', lr=1.0, n_iter=1000, conv_tol=1e-10, conv_window=5)

res_mv = minimize_spread(U_init, Mmn_opt, wdata.wb, wdata.bvecs, wdata.kb_idx, **COMMON)
res_ss = minimize_spread(U_init, Mmn_opt, wdata.wb, wdata.bvecs, wdata.kb_idx,
                        use_ss_functional=True, **COMMON)
res_pm = minimize_spread(U_init, Mmn_opt, wdata.wb, wdata.bvecs, wdata.kb_idx,
                        use_pm_functional=True, Aat=Aat_sub, atom_index=atom_index, **COMMON)

Q_mv = compute_pm_spread(res_mv.U_final, Aat_sub, atom_index)[1]
Q_ss = compute_pm_spread(res_ss.U_final, Aat_sub, atom_index)[1]
Q_pm = compute_pm_spread(res_pm.U_final, Aat_sub, atom_index)[1]

print(f'{"functional":6s} {"Omega (Ang^2)":>14s} {"mean max(Q)":>12s}')
for name, res, Q in [('MV', res_mv, Q_mv), ('SS', res_ss, Q_ss), ('PM', res_pm, Q_pm)]:
    omega_ang2 = res.Omega * BOHR_TO_ANG**2
    print(f'{name:6s} {omega_ang2:14.4f} {Q.max(dim=1).values.mean().item():12.4f}')

### 4. Mulliken charges Q_A[n] per functional

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(12, 4), dpi=150, sharey=True)
for a, (name, Q) in zip(ax, [('MV', Q_mv), ('SS', Q_ss), ('PM', Q_pm)]):
    im = a.imshow(Q.detach().numpy(), aspect='auto', cmap='viridis', vmin=0, vmax=1)
    a.set_title(name); a.set_xlabel('atom'); a.set_xticks([0, 1])
    fig.colorbar(im, ax=a, shrink=0.8, label='$Q_A[n]$')
ax[0].set_ylabel('Wannier function $n$')
fig.tight_layout()

**Takeaway.** MV and SS (near-identical $\Omega_I$/$\Omega_{OD}$, only $\Omega_D$'s formula differs) give very similar Mulliken charge patterns and total spread — SS is a genuine alternative $\Omega_D$ definition, not a dramatically different gauge in cases like this one without severe branch-cut pathology. PM is the outlier: its own optimization target has nothing to do with spatial compactness, and it shows — a *worse* (higher) ordinary MV spread than either MV or SS reach, traded for Wannier functions each concentrated almost entirely on a single atom (mean max Mulliken charge noticeably closer to 1 than MV/SS's own). That's the real, textbook PM-vs-MV tradeoff (Pipek & Mezey's own 1989 paper makes the same point for molecules): maximizing atomic character and minimizing spatial spread are different objectives, and there's no guarantee the same gauge optimizes both.